# 🔥 Wildfire ResNet18 Training - 5 Giorni (120 Canali)
**Versione Finale - Bug Silente Risolto**

Benvenuti nel notebook per l'addestramento cloud del modello di previsione degli incendi boschivi.
In questa versione è stato risolto un *bug silente* riguardante il disallineamento dinamico dei canali generati dal dataloader rispetto all'architettura U-Net.

---
## 🛠️ 1. Setup dell'Ambiente e Dipendenze
Iniziamo installando le librerie necessarie (come PyTorch Lightning, W&B, e Segmentation Models) e configurando l'ambiente Colab clonando la repository originale.

In [ ]:
# Installazione pacchetti principali
!pip install -q pytorch-lightning wandb segmentation-models-pytorch h5py scikit-learn tqdm pyyaml rasterio

import os
import sys
from pathlib import Path

### Montaggio di Google Drive e Clonazione Repo

In [ ]:
# Setup Colab: Montaggio Drive
from google.colab import drive
drive.mount('/content/drive')

# Clonazione della repository ufficiale e posizionamento nella directory di lavoro
!cd /content && git clone https://github.com/SebastianGer/WildfireSpreadTS.git
os.chdir('/content/WildfireSpreadTS')

## 🔑 2. Login a Weights & Biases
Effettuiamo il login a W&B per tracciare automaticamente l'addestramento, le metriche e i log del modello in cloud.

In [ ]:
import wandb
wandb.login()

## 📂 3. Configurazione dei Percorsi (Paths)
Impostiamo le costanti per indicare dove trovare i dati su Drive e dove salvare gli output dell'addestramento. Verifichiamo inoltre quanti file HDF5 (i file di misurazione geospaziale temporale) sono effettivamente disponibili.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Wildfire_Project"
OUTPUT_DIR = "/content/lightning_logs"
CONFIGS_DIR = "/content/WildfireSpreadTS/cfgs"

# Creazione cartella output se non esiste
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Directory dei dati: {DATA_DIR}\n")
print("Struttura dei contenuti trovata:")

total_hdf5 = 0
for year in [2018, 2019, 2020, 2021]:
    year_dir = Path(DATA_DIR) / str(year)
    if year_dir.exists():
        hdf5_files = list(year_dir.glob("*.hdf5"))
        print(f"  {year}: {len(hdf5_files)} file HDF5")
        total_hdf5 += len(hdf5_files)
    else:
        print(f"  {year}: NON TROVATO")

print(f"\nTotale file HDF5 per l'addestramento: {total_hdf5}")

## ⚙️ 4. Creazione File di Configurazione (YAML)
Per lanciare l'addestramento tramite il framework `LightningCLI`, generiamo dinamicamente i file di configurazione YAML necessari.

### Configurazione del Dataloader (Data)
Impostiamo **5 giorni** di contesto temporale (`n_leading_observations: 5`) e assicuriamoci di rimuovere i canali statici duplicati per ottimizzare il volume di informazioni gestite.

In [ ]:
data_config = f"""
data_dir: {DATA_DIR}
batch_size: 16
n_leading_observations: 5
crop_side_length: 128
load_from_hdf5: true
num_workers: 4
remove_duplicate_features: true
features_to_keep: null
n_leading_observations_test_adjustment: 5
"""

# Salvataggio del config data
data_config_path = f"{CONFIGS_DIR}/data_colab_5days.yaml"
with open(data_config_path, "w") as f:
    f.write(data_config.strip())

print(f"✓ Configurazione Data creata: {data_config_path}")

### Configurazione del Trainer (PyTorch Lightning)
Specifichiamo l'uso di una singola GPU, salvataggio dei log su W&B e monitoraggio della *Validation Loss* per 100 epoche massime.

In [ ]:
trainer_config = """
accelerator: gpu
strategy: auto
devices: 1
num_nodes: 1
precision: 32-true
logger: 
  class_path: pytorch_lightning.loggers.wandb.WandbLogger
  init_args:
    project: wildfire_resnet18_5days_colab
    log_model: true
callbacks: 
  - class_path: pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint
    init_args:
      monitor: val_loss
      mode: min
max_epochs: 100
check_val_every_n_epoch: 1
enable_progress_bar: true
default_root_dir: """ + OUTPUT_DIR

# Salvataggio del config trainer
trainer_config_path = f"{CONFIGS_DIR}/trainer_colab_5days.yaml"
with open(trainer_config_path, "w") as f:
    f.write(trainer_config.strip())

print(f"✓ Configurazione Trainer creata: {trainer_config_path}")

## 🔍 5. Fase Cruciale: Calcolo del Numero Effettivo di Canali

**Problema (Bug Silente)**: Usando le configurazioni statiche dei ricercatori originali con una fusione temporale a 5 giorni e l'eliminazione dei duplicati statici, il numero reale di canali calcolato non corrisponde sempre a quello previsto teoricamente (generando un crash *Shape Mismatch* all'avvio).

**Soluzione**: Inizializziamo brevemente il `Datamodule` per estrarre il primo batch reale, leggere l'esatta dimensione della matrice canali e usare tale valore con precisione chirurgica nel config dell'encoder U-Net.

In [ ]:
import h5py
import torch
from src.dataloader.FireSpreadDataModule import FireSpreadDataModule
from src.dataloader.FireSpreadDataset import FireSpreadDataset

print("="*70)
print("FASE 1: CALCOLO DEL NUMERO REALE DI CANALI DAL BATCH")
print("="*70)

print("\n1️⃣ Inizializzazione DataModule con fusione a 5 giorni...")

datamodule = FireSpreadDataModule(
    data_dir=DATA_DIR,
    batch_size=8,
    n_leading_observations=5,  # 5 GIORNI DI STORICO
    n_leading_observations_test_adjustment=5,
    crop_side_length=128,
    load_from_hdf5=True,
    num_workers=0, # Usiamo 0 temporaneamente per caricamento sincrono immediato
    remove_duplicate_features=True  # Rimuove dati statici (come l'altitudine) ridondanti
)

# Avvio del dataloader
datamodule.setup("fit")
train_loader = datamodule.train_dataloader()

# Estrazione di un sample batch reale
sample_batch = next(iter(train_loader))
x_sample, y_sample = sample_batch

# 🔴 ESTRAZIONE DINAMICA DEL NUMERO DI CANALI DAL SENSORE
actual_n_channels = x_sample.shape[1]

print(f"\n2️⃣ Analisi del batch estratto:")
print(f"   - Forma Input (X): {x_sample.shape}")
print(f"   - Forma Output (Y): {y_sample.shape}")
print(f"\n   ✅ CANALI EFFETTIVI (n_channels): {actual_n_channels}")

# Controllo opzionale con il metodo statico per validazione
n_features_static = FireSpreadDataset.get_n_features(
    n_observations=5,
    features_to_keep=None,
    deduplicate_static_features=True
)

if actual_n_channels == n_features_static:
    print(f"\n   ✅ I valori combaciano con il metodo get_n_features(): {n_features_static}")
else:
    print(f"\n   ⚠️ Disallineamento rilevato! Utilizzeremo il valore empirico ({actual_n_channels}) per evitare crash.")

print(f"\n3️⃣ Statistiche Dataset:")
print(f"   - Training samples: {len(datamodule.train_dataset)}")
print(f"   - Validation samples: {len(datamodule.val_dataset)}")

## 🧠 6. Creazione Configurazione Modello
Ora che sappiamo **con precisione** che i canali sono pari alla variabile `actual_n_channels`, creiamo il file di configurazione del modello personalizzato *prima* di avviare il vero processo di training isolato. Questo bypassa totalmente il bug.

In [ ]:
print("="*70)
print("FASE 2: CREAZIONE CONFIG MODELLO SICURA")
print("="*70 + "\n")

model_config = f"""
seed_everything: 0
optimizer:
  class_path: torch.optim.AdamW
  init_args:
    lr: 0.001
model:
  class_path: models.SMPModel
  init_args:
    encoder_name: resnet18
    n_channels: {actual_n_channels}
    flatten_temporal_dimension: true
    pos_class_weight: 236
    loss_function: Dice
do_train: true
do_predict: false
do_test: false
"""

model_config_path = f"{CONFIGS_DIR}/unet/res18_5days.yaml"
with open(model_config_path, "w") as f:
    f.write(model_config.strip())

print(f"✓ Configurazione Modello creata: {model_config_path}")
print(f"  n_channels impostato hard-coded a: {actual_n_channels}")
print(f"\n🎯 Il bug silente è prevenuto: il nuovo processo di training leggerà questo file corretto.")

## ✅ 7. Verifica Architettura Modello (Forward Pass Test)
A fini di debug e per conferma assoluta, inizializziamo il modello puro e proviamo un *"forward pass"* con un tensore dummy che simula i nostri dati. Se passa questo step, il training filerà liscio.

In [ ]:
from src.models.SMPModel import SMPModel

print("="*70)
print("FASE 3: VERIFICA ARCHITETTURA MODELLO (DRY-RUN)")
print("="*70 + "\n")

model = SMPModel(
    encoder_name="resnet18",
    n_channels=actual_n_channels,
    flatten_temporal_dimension=True,
    pos_class_weight=236,
    loss_function="Dice"
)

print(f"✓ Modello inizializzato: U-Net con ResNet18 ({actual_n_channels} canali input)")

with torch.no_grad():
    # Creazione di un tensore di prova (Batch 2, Channels X, 128x128)
    x_test = torch.randn(2, actual_n_channels, 128, 128)
    y_test = model(x_test)
    
    print(f"✓ Forward pass test superato con successo!")
    print(f"  - Output shape prevista (2, 1, 128, 128): {y_test.shape}")

## 🚀 8. Avvio dell'Addestramento (Main Training)
Ora che abbiamo isolato tutte le configurazioni e risolto i conflitti, avviamo il training vero e proprio. 

Utilizziamo `os.system` in modo che PyTorch Lightning possa inizializzare i processi CUDA in modo nativo su una sessione pulita e separata tramite il suo modulo `LightningCLI`, evitando memory leak del notebook.

In [ ]:
print("="*80)
print("FASE 4: AVVIO DEL TRAINING")
print("="*80 + "\n")

training_command = f"""
cd /content/WildfireSpreadTS && \
python src/train.py \
  -c cfgs/unet/res18_5days.yaml \
  --trainer cfgs/trainer_colab_5days.yaml \
  --data cfgs/data_colab_5days.yaml \
  --do_train true \
  --do_test false \
  --do_validate false \
  --trainer.default_root_dir {OUTPUT_DIR}
"""

print("Esecuzione comando di training CLI:\n")
print(training_command.strip())
print("\nAvvio del subprocess in corso...")

# Avvio del subprocess
os.system(training_command)

print("\n" + "="*80)
print("✓ ADDESTRAMENTO COMPLETATO!")
print("="*80)

## 📦 9. Recupero del Miglior Modello (Checkpoint)
Dopo il training, il callback `ModelCheckpoint` avrà salvato i file fisici del modello. Recuperiamo l'ultimo checkpoint generato per averlo a disposizione.

In [ ]:
checkpoint_dir = Path(OUTPUT_DIR)
checkpoints = list(checkpoint_dir.glob("**/checkpoints/*.ckpt"))

if checkpoints:
    # Prendi il file modificato più di recente
    best_checkpoint = sorted(checkpoints, key=lambda x: x.stat().st_mtime)[-1]
    print(f"✓ Miglior checkpoint individuato:")
    print(f"  Percorso: {best_checkpoint}")
    print(f"  Dimensione: {best_checkpoint.stat().st_size / 1e6:.2f} MB")
else:
    print(f"✗ Nessun checkpoint trovato. Si è verificato un problema nel salvataggio.")

## 📊 10. Riepilogo Addestramento e Fixes
Una panoramica riassuntiva di ciò che abbiamo appena computato.

In [ ]:
print("="*80)
print("🎉 ADDESTRAMENTO CON FUSIONE TEMPORALE A 5 GIORNI COMPLETATO!")
print("="*80)

print(f"\n📊 RIEPILOGO STATISTICHE")
print(f"─" * 80)
print(f"Modello: U-Net con encoder ResNet18")
print(f"Contesto temporale in Input: 5 giorni consecutivi")
print(f"Canali in Input reali: {actual_n_channels} (dinamici + statici de-duplicati)")
print(f"Risoluzione spaziale in addestramento: 128×128 pixel")
print(f"Batch size: 16")
print(f"Epoche max: 100")

print(f"\n🐛 BUG FIX APPLICATI")
print(f"─" * 80)
print(f"✅ Bug Silente Prevenuto:")
print(f"   - Numero canali calcolato dinamicamente: n_channels={actual_n_channels}")
print(f"   - Inserito con successo all'interno di res18_5days.yaml prima del lancio CLI")
print(f"   - Disallineamento tensori e Shape Mismatch evitati!")

print(f"\n📈 PROSSIMI PASSI")
print(f"─" * 80)
print(f"1. Monitorare le metriche online su Weights & Biases")
print(f"2. Assicurarsi del corretto backup su Google Drive")
print(f"3. Validazione tramite export in ONNX")
print("\n" + "="*80)

## 💾 11. Salvataggio su Google Drive
Per prevenire la perdita dei dati alla chiusura della sessione Colab temporanea, trasferiamo il checkpoint pesato e i log (necessari per W&B o Tensorboard) in via definitiva sul nostro spazio *Google Drive* montato in precedenza.

In [ ]:
import shutil

drive_output = "/content/drive/MyDrive/Wildfire_Project/training_results_5days"
Path(drive_output).mkdir(parents=True, exist_ok=True)

try:
    if checkpoints:
        # Copia checkpoint
        checkpoint_dest = f"{drive_output}/checkpoints"
        Path(checkpoint_dest).mkdir(parents=True, exist_ok=True)
        for ckpt in checkpoints:
            shutil.copy2(ckpt, f"{checkpoint_dest}/{ckpt.name}")
        print(f"✓ Checkpoints copiati con successo in:\n  {checkpoint_dest}")
    
    # Copia logs Lightning
    logs_dest = f"{drive_output}/lightning_logs"
    shutil.copytree(OUTPUT_DIR, logs_dest, dirs_exist_ok=True)
    print(f"\n✓ Log salvati in:\n  {logs_dest}")
    print(f"\n🎉 Tutti i risultati sono al sicuro su Google Drive.")
except Exception as e:
    print(f"⚠️ Errore durante il salvataggio su Drive: {e}")

## 🛰️ 12. Compilazione e Ottimizzazione del Grafo ONNX (Uso Spaziale)
Ora che abbiamo il modello addestrato, dobbiamo esportarlo nel formato universale `ONNX`. L'architettura U-Net non girerà sui server cloud, bensì a bordo del satellite in Low-Earth Orbit dove le risorse energetiche e computazionali sono fortemente limitate. 

In questa fase andiamo a staticizzare le dimensioni (risoluzione $64\times64$ richiesta dalla telecamera del satellite) e a fissare lo stato del modello per la *pure inference* (es. blocchiamo BatchNorm e Dropout).

In [ ]:
print("="*80)
print("⚙️ ESPORTAZIONE ONNX PER INFERENZA SATELLITARE IN ORBITA")
print("="*80 + "\n")

if checkpoints:
    print("Avvio compilazione del grafo ONNX per il satellite...")
    
    # Caricamento Modello dal Miglior Checkpoint trovato
    # Sfruttiamo il "actual_n_channels" calcolato dinamicamente nelle celle precedenti!
    model_onnx = SMPModel.load_from_checkpoint(
        str(best_checkpoint),
        encoder_name="resnet18",
        n_channels=actual_n_channels,
        flatten_temporal_dimension=True
    )
    
    # Modalità Valutazione: essenziale per bloccare l'aggiornamento pesi, BatchNorm e Dropout
    model_onnx.eval()
    model_onnx.to('cpu') 
    
    # Tensore dummy per ricalcare ESATTAMENTE lo spazio vettoriale in input al satellite.
    # L'architettura di TinySML richiederà una finestra statica 64x64.
    dummy_satellite_input = torch.randn(1, actual_n_channels, 64, 64)
    
    # Destinazione finale per il recupero
    onnx_dest_path = "/content/drive/MyDrive/Wildfire_Project/wsts_model.onnx"
    
    # Tracciatura del modello e compilazione in un grafo statico computazionale 
    torch.onnx.export(
        model_onnx,                      
        dummy_satellite_input,                
        onnx_dest_path,             
        export_params=True,         
        opset_version=14,           
        do_constant_folding=True,  
        input_names=['input'],      
        output_names=['output'],    
        dynamic_axes={
            'input': {0: 'batch_size'}, 
            'output': {0: 'batch_size'}
        }
    )
    
    print(f"\n🎉 MISSIONE COMPIUTA: Modello Spaziale ONNX esportato direttamente su Drive!")
    print(f"Posizione: {onnx_dest_path}")
else:
    print("Esportazione ONNX annullata: modello di base / checkpoint non trovato.")